In [1]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

# Configuración obligatoria para mostrar gráficos en celdas de Jupyter
pio.renderers.default = "notebook"
print("✓ Librerías cargadas correctamente.")

✓ Librerías cargadas correctamente.


In [2]:
def obtener_datos(ticker_symbol, periodo):
    """Obtiene datos históricos desde la API pública de yfinance."""
    try:
        ticker = yf.Ticker(ticker_symbol)
        df = ticker.history(period=periodo)
        
        if df.empty:
            raise ValueError(f"No se encontraron datos para {ticker_symbol}")
            
        # Métricas calculadas
        df['MA_20'] = df['Close'].rolling(window=20).mean()
        df['MA_50'] = df['Close'].rolling(window=50).mean()
        return df
    except Exception as e:
        print(f"⚠️ Error al conectar con la API: {e}")
        return None

In [3]:
# Elementos de la interfaz de usuario (Widgets)
ticker_widget = widgets.Dropdown(
    options=['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'TSLA'],
    value='AAPL',
    description='Acción:'
)

periodo_widget = widgets.ToggleButtons(
    options=['1m', '3m', '6m', '1y', '2y'],
    value='1y',
    description='Periodo:',
    button_style='info'
)

out = widgets.Output()

def actualizar_dashboard(change=None):
    df = obtener_datos(ticker_widget.value, periodo_widget.value)
    
    with out:
        out.clear_output(wait=True)
        if df is None:
            return
            
        fig = make_subplots(
            rows=2, cols=1, 
            shared_xaxes=True, 
            subplot_titles=(f'Cotización de {ticker_widget.value}', 'Volumen de Negociación'),
            row_width=[0.3, 0.7]
        )

        # Gráfico de Velas Japonesas
        fig.add_trace(go.Candlestick(
            x=df.index, open=df['Open'], high=df['High'],
            low=df['Low'], close=df['Close'], name='Precio'
        ), row=1, col=1)

        # Medias Móviles
        fig.add_trace(go.Scatter(x=df.index, y=df['MA_20'], line=dict(color='orange', width=1), name='Media 20d'), row=1, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['MA_50'], line=dict(color='blue', width=1), name='Media 50d'), row=1, col=1)

        # Volumen
        fig.add_trace(go.Bar(x=df.index, y=df['Volume'], name='Volumen', marker_color='teal'), row=2, col=1)

        fig.update_layout(height=550, template="plotly_white", xaxis_rangeslider_visible=False)
        fig.show()

# Vincular actualización al modificar controles
ticker_widget.observe(actualizar_dashboard, names='value')
periodo_widget.observe(actualizar_dashboard, names='value')

# Renderizar Dashboard
display(widgets.HBox([ticker_widget, periodo_widget]), out)
actualizar_dashboard()

Output()